In [2]:
#调用模型
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
model = ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=10,max_tokens=1000)

## 自定义中间件
#### 通过实现在代理执行流中的特定点运行的挂钩来构建自定义中间件。
#### 您可以通过两种方式创建中间件：
#### (1)、基于装饰器 - 对于单钩中间件来说快速而简单
#### (2)、基于类 - 对于具有多个钩子的复杂中间件来说更强大
### 1、基于装饰器
#### 基于装饰器的自定义中间件我有写在short-term momery的最后部分

In [ ]:
from langchain.agents.middleware import before_model, after_model, wrap_model_call
from langchain.agents.middleware import AgentState, ModelRequest, ModelResponse, dynamic_prompt
from langchain.messages import AIMessage
from langchain.agents import create_agent
from langgraph.runtime import Runtime
from typing import Any, Callable


# Node-style: logging before model calls
@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"About to call model with {len(state['messages'])} messages")
    return None

# Node-style: validation after model calls
@after_model(can_jump_to=["end"])
def validate_output(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    last_message = state["messages"][-1]
    if "BLOCKED" in last_message.content:
        return {
            "messages": [AIMessage("I cannot respond to that request.")],
            "jump_to": "end"
        }
    return None

# Wrap-style: retry logic
@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            if attempt == 2:
                raise
            print(f"Retry {attempt + 1}/3 after error: {e}")

# Wrap-style: dynamic prompts
@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    user_id = request.runtime.context.get("user_id", "guest")
    return f"You are a helpful assistant for user {user_id}. Be concise and friendly."

# Use decorators in agent
agent = create_agent(
    model="gpt-4o",
    middleware=[log_before_model, validate_output, retry_model, personalized_prompt],
    tools=[...],
)

#### 可用的装饰器
##### 节点样式（在特定执行点运行）：
##### ① @before_agent- 代理启动前（每次调用一次）
##### ② @before_model - 每次模型调用之前
##### ③ @after_model - 在每个模型响应之后
##### ④ @after_agent- 代理完成后（每次调用一次）
##### 包装式（拦截和控制执行）：
##### ① @wrap_model_call - 围绕每个模型调用
##### ② @wrap_tool_call - 围绕每个工具调用
##### 便利装饰器：
##### ① @dynamic_prompt - 生成动态系统提示（相当于修改提示的@wrap_model_call）

### 2、基于类的中间件
#### 分为Node-style hooks（节点式 / 生命周期式）和Wrap-style hooks（包裹式 / 环绕式）
#### （1）Node-style hooks（节点式 / 生命周期式）
##### 在执行流中的特定点运行，其函数有：
##### ① before_agent- 代理启动前（每次调用一次）
##### ② before_model- 每次模型调用之前
##### ③ after_model- 在每个模型响应之后
##### ④ after_agent- 代理完成后（每次调用最多一次）

In [ ]:
# 示例：日志记录中间件
from langchain.agents.middleware import AgentMiddleware, AgentState
from langgraph.runtime import Runtime
from typing import Any

class LoggingMiddleware(AgentMiddleware):
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"About to call model with {len(state['messages'])} messages")
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"Model returned: {state['messages'][-1].content}")
        return None

#### （2）Wrap-style hooks（包裹式 / 环绕式）
##### 调用处理程序时拦截执行和控制：
##### ① wrap_model_call- 围绕每个模型调用
##### ② wrap_tool_call- 围绕每个工具调用
##### 您可以决定处理程序是调用零次（短路）、一次（正常流）还是多次（重试逻辑）。

In [ ]:
# 示例：模型重试中间件
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from typing import Callable

class RetryMiddleware(AgentMiddleware):
    def __init__(self, max_retries: int = 3):
        super().__init__()
        self.max_retries = max_retries

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        for attempt in range(self.max_retries):
            try:
                return handler(request)
            except Exception as e:
                if attempt == self.max_retries - 1:
                    raise
                print(f"Retry {attempt + 1}/{self.max_retries} after error: {e}")